In [2]:
# ============================================================
# BAKERY SALES & MARGIN INTELLIGENCE
# STAGE 2 — DATA CLEANING & PREPARATION
# STEP 1: DATA QUALITY DIAGNOSIS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. PROJECT PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

print("=" * 70)
print("STAGE 2 — DATA QUALITY DIAGNOSIS")
print("=" * 70)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Data folder : {DATA_DIR}")


# ============================================================
# 2. LOAD DATASETS
# ============================================================

transactions = pd.read_csv(
    DATA_DIR / "transactions_raw.csv"
)

products = pd.read_csv(
    DATA_DIR / "products.csv"
)

production_log = pd.read_csv(
    DATA_DIR / "production_log.csv"
)

cost_changes = pd.read_csv(
    DATA_DIR / "cost_changes.csv"
)

price_changes = pd.read_csv(
    DATA_DIR / "price_changes.csv"
)

holidays = pd.read_csv(
    DATA_DIR / "holidays.csv"
)


# ============================================================
# 3. CREATE WORKING COPIES
# ============================================================

transactions_clean = transactions.copy()
products_clean = products.copy()
production_clean = production_log.copy()
cost_changes_clean = cost_changes.copy()
price_changes_clean = price_changes.copy()
holidays_clean = holidays.copy()


# ============================================================
# 4. DATASET SUMMARY
# ============================================================

datasets = {
    "transactions": transactions_clean,
    "products": products_clean,
    "production_log": production_clean,
    "cost_changes": cost_changes_clean,
    "price_changes": price_changes_clean,
    "holidays": holidays_clean
}

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

for name, df in datasets.items():

    print(
        f"{name:<20} "
        f"{len(df):>8,} rows × "
        f"{len(df.columns):>3} columns"
    )


# ============================================================
# 5. TRANSACTION STRUCTURE
# ============================================================

print("\n" + "=" * 70)
print("2.1 TRANSACTIONS STRUCTURE")
print("=" * 70)

print("\nColumns:")
print(list(transactions_clean.columns))

print("\nData types:")

display(
    transactions_clean.dtypes.to_frame("dtype")
)


# ============================================================
# 6. TRANSACTION MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("2.2 TRANSACTION MISSING VALUES")
print("=" * 70)

transaction_missing = pd.DataFrame({
    "missing_count": transactions_clean.isna().sum(),
    "missing_percent": (
        transactions_clean.isna().mean() * 100
    ).round(2)
})

transaction_missing = transaction_missing[
    transaction_missing["missing_count"] > 0
]

if transaction_missing.empty:

    print("No missing values found.")

else:

    display(transaction_missing)


# ============================================================
# 7. EXAMPLES OF INCOMPLETE TRANSACTIONS
# ============================================================

print("\n" + "=" * 70)
print("2.3 INCOMPLETE TRANSACTION EXAMPLES")
print("=" * 70)

incomplete_transactions = transactions_clean[
    transactions_clean.isna().any(axis=1)
]

print(
    f"Rows containing missing values: "
    f"{len(incomplete_transactions):,}"
)

display(
    incomplete_transactions.head(10)
)


# ============================================================
# 8. DUPLICATE TRANSACTIONS
# ============================================================

print("\n" + "=" * 70)
print("2.4 DUPLICATE TRANSACTIONS")
print("=" * 70)

duplicate_count = transactions_clean.duplicated().sum()

print(
    f"Duplicate rows: {duplicate_count:,}"
)

if duplicate_count > 0:

    duplicate_transactions = transactions_clean[
        transactions_clean.duplicated(keep=False)
    ]

    display(
        duplicate_transactions.head(20)
    )


# ============================================================
# 9. TRANSACTION ID CHECK
# ============================================================

print("\n" + "=" * 70)
print("2.5 TRANSACTION ID CHECK")
print("=" * 70)

possible_id_columns = [
    col for col in transactions_clean.columns
    if any(
        word in col.lower()
        for word in [
            "transaction_id",
            "transactionid",
            "sale_id",
            "order_id",
            "id"
        ]
    )
]

print(
    "Possible ID columns:",
    possible_id_columns
)

for column in possible_id_columns:

    print(f"\n{column}")

    print(
        "Unique values:",
        transactions_clean[column].nunique()
    )

    print(
        "Missing:",
        transactions_clean[column].isna().sum()
    )

    print(
        "Duplicate IDs:",
        transactions_clean[column].duplicated().sum()
    )


# ============================================================
# 10. NUMERICAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("2.6 TRANSACTION NUMERICAL VALIDATION")
print("=" * 70)

numeric_columns = transactions_clean.select_dtypes(
    include=np.number
).columns.tolist()

print("Numeric columns:", numeric_columns)

for column in numeric_columns:

    negative_count = (
        transactions_clean[column] < 0
    ).sum()

    zero_count = (
        transactions_clean[column] == 0
    ).sum()

    print(f"\n{column}")

    print(
        f"  Minimum          : "
        f"{transactions_clean[column].min()}"
    )

    print(
        f"  Maximum          : "
        f"{transactions_clean[column].max()}"
    )

    print(
        f"  Negative values  : "
        f"{negative_count:,}"
    )

    print(
        f"  Zero values      : "
        f"{zero_count:,}"
    )


# ============================================================
# 11. PRODUCT ID VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("2.7 PRODUCT ID VALIDATION")
print("=" * 70)

print("\nTransaction columns:")
print(list(transactions_clean.columns))

print("\nProduct columns:")
print(list(products_clean.columns))

transaction_product_columns = [
    col for col in transactions_clean.columns
    if "product" in col.lower()
]

product_product_columns = [
    col for col in products_clean.columns
    if "product" in col.lower()
]

print(
    "\nTransaction product columns:",
    transaction_product_columns
)

print(
    "Product table product columns:",
    product_product_columns
)


# ============================================================
# 12. PRODUCTION LOG VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("2.8 PRODUCTION LOG VALIDATION")
print("=" * 70)

print("\nColumns:")
print(list(production_clean.columns))

print("\nMissing values:")

display(
    production_clean.isna().sum().to_frame(
        "missing_count"
    )
)

print("\nSample:")

display(
    production_clean.head(10)
)


# ============================================================
# 13. PRODUCTION NUMERICAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("2.9 PRODUCTION NUMERICAL CHECK")
print("=" * 70)

production_numeric = production_clean.select_dtypes(
    include=np.number
).columns.tolist()

print(
    "Numeric columns:",
    production_numeric
)

for column in production_numeric:

    print(f"\n{column}")

    print(
        f"  Minimum : "
        f"{production_clean[column].min()}"
    )

    print(
        f"  Maximum : "
        f"{production_clean[column].max()}"
    )

    print(
        f"  Negative: "
        f"{(production_clean[column] < 0).sum():,}"
    )


# ============================================================
# 14. SUPPORTING DATASETS
# ============================================================

print("\n" + "=" * 70)
print("2.10 SUPPORTING DATASETS")
print("=" * 70)

print("\nPRODUCTS")
display(products_clean)

print("\nCOST CHANGES")
display(cost_changes_clean)

print("\nPRICE CHANGES")
display(price_changes_clean)

print("\nHOLIDAYS")
display(holidays_clean)


# ============================================================
# 15. FINAL DIAGNOSIS SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STAGE 2 DIAGNOSIS SUMMARY")
print("=" * 70)

print(
    f"\nTransactions       : {len(transactions_clean):,}"
)

print(
    f"Transaction missing: "
    f"{transactions_clean.isna().sum().sum():,}"
)

print(
    f"Transaction dupes  : "
    f"{transactions_clean.duplicated().sum():,}"
)

print(
    f"Production records : "
    f"{len(production_clean):,}"
)

print(
    f"Production missing : "
    f"{production_clean.isna().sum().sum():,}"
)

print("\nNo records have been deleted or modified.")

print("\nSTAGE 2 DIAGNOSIS COMPLETE.")

STAGE 2 — DATA QUALITY DIAGNOSIS

Project root: c:\Users\user\Desktop\Bakery dashboard
Data folder : c:\Users\user\Desktop\Bakery dashboard\data

DATASET SUMMARY
transactions           70,720 rows ×   8 columns
products                   20 rows ×   5 columns
production_log          3,680 rows ×   4 columns
cost_changes                7 rows ×   3 columns
price_changes               2 rows ×   3 columns
holidays                    6 rows ×   2 columns

2.1 TRANSACTIONS STRUCTURE

Columns:
['transaction_id', 'timestamp', 'product_name', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'cashier']

Data types:


,dtype
transaction_id,object
timestamp,object
product_name,object
quantity,int64
unit_price,float64
discount_pct,float64
payment_method,object
cashier,object



2.2 TRANSACTION MISSING VALUES


,missing_count,missing_percent
unit_price,260,0.37
discount_pct,3350,4.74
cashier,2095,2.96



2.3 INCOMPLETE TRANSACTION EXAMPLES
Rows containing missing values: 5,598


...


PRICE CHANGES


,product_name,effective_date,new_list_price
0,Agege Bread,2026-07-01,1400
1,Sliced Sandwich Bread,2026-07-15,2000



HOLIDAYS


,date,holiday
0,2026-03-20,Eid al-Fitr
1,2026-04-03,Good Friday
2,2026-04-06,Easter Monday
3,2026-05-01,Workers Day
4,2026-05-27,Eid al-Adha
5,2026-06-12,Democracy Day



STAGE 2 DIAGNOSIS SUMMARY

Transactions       : 70,720
Transaction missing: 5,705
Transaction dupes  : 620
Production records : 3,680
Production missing : 343

No records have been deleted or modified.

STAGE 2 DIAGNOSIS COMPLETE.


In [3]:
# ============================================================
# BAKERY SALES & MARGIN INTELLIGENCE
# STAGE 2 — DATA CLEANING & PREPARATION
# STEP 2: CLEANING PIPELINE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STAGE 2 — DATA CLEANING PIPELINE")
print("=" * 70)

print(f"\nRaw data      : {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")


# ============================================================
# 2. LOAD RAW DATA
# ============================================================

transactions = pd.read_csv(
    RAW_DIR / "transactions_raw.csv"
)

products = pd.read_csv(
    RAW_DIR / "products.csv"
)

production_log = pd.read_csv(
    RAW_DIR / "production_log.csv"
)

cost_changes = pd.read_csv(
    RAW_DIR / "cost_changes.csv"
)

price_changes = pd.read_csv(
    RAW_DIR / "price_changes.csv"
)

holidays = pd.read_csv(
    RAW_DIR / "holidays.csv"
)


# ============================================================
# 3. CREATE CLEANING COPIES
# ============================================================

transactions_clean = transactions.copy()
products_clean = products.copy()
production_clean = production_log.copy()
cost_changes_clean = cost_changes.copy()
price_changes_clean = price_changes.copy()
holidays_clean = holidays.copy()


# ============================================================
# 4. STANDARDIZE COLUMN NAMES
# ============================================================

def clean_column_names(df):

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    return df


transactions_clean = clean_column_names(
    transactions_clean
)

products_clean = clean_column_names(
    products_clean
)

production_clean = clean_column_names(
    production_clean
)

cost_changes_clean = clean_column_names(
    cost_changes_clean
)

price_changes_clean = clean_column_names(
    price_changes_clean
)

holidays_clean = clean_column_names(
    holidays_clean
)


# ============================================================
# 5. STANDARDIZE TRANSACTION TEXT FIELDS
# ============================================================

text_columns = [
    "transaction_id",
    "product_name",
    "payment_method",
    "cashier"
]

for column in text_columns:

    if column in transactions_clean.columns:

        transactions_clean[column] = (
            transactions_clean[column]
            .astype("string")
            .str.strip()
        )


# ============================================================
# 6. STANDARDIZE PRODUCT NAMES
# ============================================================

if "product_name" in products_clean.columns:

    products_clean["product_name"] = (
        products_clean["product_name"]
        .astype("string")
        .str.strip()
    )

if "product_name" in production_clean.columns:

    production_clean["product_name"] = (
        production_clean["product_name"]
        .astype("string")
        .str.strip()
    )


# ============================================================
# 7. STANDARDIZE PAYMENT METHODS
# ============================================================

if "payment_method" in transactions_clean.columns:

    transactions_clean["payment_method"] = (
        transactions_clean["payment_method"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    payment_mapping = {
        "pos": "POS",
        "cash": "Cash",
        "transfer": "Transfer"
    }

    transactions_clean["payment_method"] = (
        transactions_clean["payment_method"]
        .replace(payment_mapping)
    )


# ============================================================
# 8. CONVERT TRANSACTION TIMESTAMP
# ============================================================

if "timestamp" in transactions_clean.columns:

    transactions_clean["timestamp"] = pd.to_datetime(
        transactions_clean["timestamp"],
        errors="coerce"
    )


# ============================================================
# 9. CONVERT PRODUCTION DATE
# ============================================================

if "date" in production_clean.columns:

    production_clean["date"] = pd.to_datetime(
        production_clean["date"],
        errors="coerce"
    )


# ============================================================
# 10. CONVERT OTHER DATE COLUMNS
# ============================================================

for df in [
    cost_changes_clean,
    price_changes_clean,
    holidays_clean
]:

    for column in df.columns:

        if "date" in column.lower():

            df[column] = pd.to_datetime(
                df[column],
                errors="coerce"
            )


# ============================================================
# 11. NUMERICAL DATA TYPES
# ============================================================

transaction_numeric = [
    "quantity",
    "unit_price",
    "discount_pct"
]

for column in transaction_numeric:

    if column in transactions_clean.columns:

        transactions_clean[column] = pd.to_numeric(
            transactions_clean[column],
            errors="coerce"
        )


production_numeric = [
    "units_produced",
    "units_wasted_recorded"
]

for column in production_numeric:

    if column in production_clean.columns:

        production_clean[column] = pd.to_numeric(
            production_clean[column],
            errors="coerce"
        )


# ============================================================
# 12. INVESTIGATE NEGATIVE QUANTITIES
# ============================================================

print("\n" + "=" * 70)
print("NEGATIVE QUANTITY INVESTIGATION")
print("=" * 70)

negative_quantity = transactions_clean[
    transactions_clean["quantity"] < 0
].copy()

print(
    f"Negative quantity records: "
    f"{len(negative_quantity):,}"
)

if len(negative_quantity) > 0:

    print("\nNegative quantity summary:")

    display(
        negative_quantity[
            [
                "transaction_id",
                "timestamp",
                "product_name",
                "quantity",
                "unit_price"
            ]
        ].head(20)
    )


# ============================================================
# 13. HANDLE NEGATIVE QUANTITIES
# ============================================================
#
# IMPORTANT:
# We do NOT automatically delete these.
#
# Negative sales quantities may represent returns,
# corrections, or data-entry errors.
#
# For now we flag them so they can be handled explicitly
# in the business logic.
# ============================================================

transactions_clean["is_negative_quantity"] = (
    transactions_clean["quantity"] < 0
)

print(
    f"\nNegative quantities flagged: "
    f"{transactions_clean['is_negative_quantity'].sum():,}"
)


# ============================================================
# 14. REMOVE EXACT DUPLICATE ROWS
# ============================================================

print("\n" + "=" * 70)
print("EXACT DUPLICATE REMOVAL")
print("=" * 70)

before_duplicates = len(transactions_clean)

transactions_clean = (
    transactions_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

after_duplicates = len(transactions_clean)

removed_duplicates = (
    before_duplicates - after_duplicates
)

print(
    f"Rows before : {before_duplicates:,}"
)

print(
    f"Rows after  : {after_duplicates:,}"
)

print(
    f"Duplicates removed: {removed_duplicates:,}"
)


# ============================================================
# 15. HANDLE MISSING DISCOUNTS
# ============================================================

print("\n" + "=" * 70)
print("MISSING DISCOUNT HANDLING")
print("=" * 70)

missing_discount_before = (
    transactions_clean["discount_pct"]
    .isna()
    .sum()
)

print(
    f"Missing discounts: "
    f"{missing_discount_before:,}"
)

# Business assumption:
# A missing discount means no discount was recorded.
#
# Therefore we use 0%.

transactions_clean["discount_pct"] = (
    transactions_clean["discount_pct"]
    .fillna(0)
)

print(
    "Missing discounts replaced with 0%."
)


# ============================================================
# 16. VALIDATE DISCOUNT RANGE
# ============================================================

invalid_discount = transactions_clean[
    (transactions_clean["discount_pct"] < 0) |
    (transactions_clean["discount_pct"] > 100)
]

print(
    f"Invalid discount values: "
    f"{len(invalid_discount):,}"
)

if len(invalid_discount) > 0:

    display(
        invalid_discount[
            [
                "transaction_id",
                "discount_pct"
            ]
        ].head(20)
    )


# ============================================================
# 17. MISSING UNIT PRICE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("MISSING UNIT PRICE")
print("=" * 70)

missing_price = transactions_clean[
    transactions_clean["unit_price"].isna()
]

print(
    f"Missing unit prices: "
    f"{len(missing_price):,}"
)

if len(missing_price) > 0:

    print("\nProducts affected:")

    display(
        missing_price["product_name"]
        .value_counts()
        .to_frame("missing_price_count")
    )


# ============================================================
# 18. FIND PRICE COLUMN IN PRODUCT TABLE
# ============================================================

possible_price_columns = [
    column
    for column in products_clean.columns
    if any(
        word in column.lower()
        for word in [
            "price",
            "selling_price",
            "unit_price"
        ]
    )
]

print(
    "\nPossible product price columns:",
    possible_price_columns
)


# ============================================================
# 19. FILL MISSING UNIT PRICES FROM PRODUCT TABLE
# ============================================================

if (
    "product_name" in products_clean.columns
    and len(possible_price_columns) > 0
):

    product_price_column = possible_price_columns[0]

    price_lookup = (
        products_clean[
            [
                "product_name",
                product_price_column
            ]
        ]
        .drop_duplicates(
            subset=["product_name"]
        )
    )

    price_lookup = price_lookup.set_index(
        "product_name"
    )[product_price_column]

    missing_before = (
        transactions_clean["unit_price"]
        .isna()
        .sum()
    )

    transactions_clean["unit_price"] = (
        transactions_clean["unit_price"]
        .fillna(
            transactions_clean["product_name"]
            .map(price_lookup)
        )
    )

    missing_after = (
        transactions_clean["unit_price"]
        .isna()
        .sum()
    )

    print(
        f"\nMissing prices before: "
        f"{missing_before:,}"
    )

    print(
        f"Missing prices after : "
        f"{missing_after:,}"
    )


# ============================================================
# 20. PRODUCTION WASTE
# ============================================================

print("\n" + "=" * 70)
print("PRODUCTION WASTE")
print("=" * 70)

missing_waste = (
    production_clean["units_wasted_recorded"]
    .isna()
    .sum()
)

print(
    f"Missing waste records: "
    f"{missing_waste:,}"
)

# We do NOT silently assume missing waste = zero.
# Instead, create a flag.

production_clean["waste_missing"] = (
    production_clean["units_wasted_recorded"]
    .isna()
)

print(
    f"Waste records flagged: "
    f"{production_clean['waste_missing'].sum():,}"
)


# ============================================================
# 21. PRODUCT VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("PRODUCT VALIDATION")
print("=" * 70)

valid_products = set(
    products_clean["product_name"]
)

transaction_products = set(
    transactions_clean["product_name"]
)

unknown_products = (
    transaction_products -
    valid_products
)

print(
    f"Products in transactions: "
    f"{len(transaction_products)}"
)

print(
    f"Products in product table: "
    f"{len(valid_products)}"
)

print(
    f"Unknown transaction products: "
    f"{len(unknown_products)}"
)

if unknown_products:

    print("\nUnknown products:")

    for product in sorted(unknown_products):

        print(" -", product)


# ============================================================
# 22. PRODUCTION PRODUCT VALIDATION
# ============================================================

production_products = set(
    production_clean["product_name"]
)

unknown_production_products = (
    production_products -
    valid_products
)

print(
    f"\nUnknown production products: "
    f"{len(unknown_production_products)}"
)

if unknown_production_products:

    for product in sorted(
        unknown_production_products
    ):

        print(" -", product)


# ============================================================
# 23. FINAL DATA QUALITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL CLEANING CHECK")
print("=" * 70)

print("\nTRANSACTIONS")

print(
    "Rows:",
    f"{len(transactions_clean):,}"
)

print(
    "Missing values:",
    f"{transactions_clean.isna().sum().sum():,}"
)

print(
    "Exact duplicates:",
    f"{transactions_clean.duplicated().sum():,}"
)

print(
    "Negative quantities:",
    f"{transactions_clean['is_negative_quantity'].sum():,}"
)


print("\nPRODUCTION")

print(
    "Rows:",
    f"{len(production_clean):,}"
)

print(
    "Missing values:",
    f"{production_clean.isna().sum().sum():,}"
)

print(
    "Missing waste:",
    f"{production_clean['waste_missing'].sum():,}"
)


# ============================================================
# 24. SAVE CLEANED DATASETS
# ============================================================

transactions_clean.to_csv(
    PROCESSED_DIR / "transactions_clean.csv",
    index=False
)

products_clean.to_csv(
    PROCESSED_DIR / "products_clean.csv",
    index=False
)

production_clean.to_csv(
    PROCESSED_DIR / "production_log_clean.csv",
    index=False
)

cost_changes_clean.to_csv(
    PROCESSED_DIR / "cost_changes_clean.csv",
    index=False
)

price_changes_clean.to_csv(
    PROCESSED_DIR / "price_changes_clean.csv",
    index=False
)

holidays_clean.to_csv(
    PROCESSED_DIR / "holidays_clean.csv",
    index=False
)


# ============================================================
# 25. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("STAGE 2 CLEANING PIPELINE COMPLETE")
print("=" * 70)

print("\nCleaned datasets saved to:")

print(PROCESSED_DIR)

print("\nFiles created:")

for file in sorted(PROCESSED_DIR.glob("*.csv")):

    print(
        f"✓ {file.name}"
    )

STAGE 2 — DATA CLEANING PIPELINE

Raw data      : c:\Users\user\Desktop\Bakery dashboard\data
Processed data: c:\Users\user\Desktop\Bakery dashboard\data\processed

NEGATIVE QUANTITY INVESTIGATION
Negative quantity records: 292

Negative quantity summary:


,transaction_id,timestamp,product_name,quantity,unit_price
94,TX0000039,2026-03-01 10:42:35,Puff Puff Pack,-3,500.0
580,TX0000276,2026-03-02 14:20:14,Cupcake,-1,800.0
612,TX0000294,2026-03-02 15:27:57,Sausage Roll,-1,700.0
835,TX0000397,2026-03-03 11:19:21,FISH ROLL,-1,600.0
853,TX0000405,2026-03-03 12:07:02,Cupcake,-1,800.0
976,TX0000473,2026-03-03 16:35:15,Zobo Bottle,-1,700.0
1709,TX0000813,2026-03-05 19:58:26,Sliced Sandwich Bread,-2,1800.0
1948,TX0000937,2026-03-06 13:40:59,Cream Cake Slice,-1,1800.0
2141,TX0001024,2026-03-06 19:54:42,Bottled Water,-1,300.0
2223,TX0001060,2026-03-07 09:43:02,Meat Pie,-2,1000.0



Negative quantities flagged: 292

EXACT DUPLICATE REMOVAL
Rows before : 70,720
Rows after  : 70,100
Duplicates removed: 620

MISSING DISCOUNT HANDLING
Missing discounts: 3,319
Missing discounts replaced with 0%.
Invalid discount values: 0

MISSING UNIT PRICE
Missing unit prices: 260

Products affected:


,missing_price_count
product_name,
Meat Pie,27
Agege Bread,25
Sliced Sandwich Bread,20
Chicken Pie,19
Bottled Water,17
Fish Roll,17
Doughnut,17
Coconut Bread,14
Cupcake,13



Possible product price columns: ['list_price_start']

Missing prices before: 260
Missing prices after : 14

PRODUCTION WASTE
Missing waste records: 343
Waste records flagged: 343

PRODUCT VALIDATION
Products in transactions: 84
Products in product table: 20
Unknown transaction products: 64

Unknown products:
 - AGEGE BREAD
 - Agege  Bread
 - BANANA CAKE LOAF
 - BIRTHDAY CAKE 8IN
 - BOTTLED WATER
 - Banana  Cake  Loaf
 - Birthday  Cake  8in
 - Bottled  Water
 - CHICKEN PIE
 - CHIN CHIN PACK
 - COCONUT BREAD
 - CREAM CAKE SLICE
 - CUPCAKE
 - Chicken  Pie
 - Chiken Pie
 - Chin  Chin  Pack
 - Coconut  Bread
 - Cream  Cake  Slice
 - DOUGHNUT
 - Donut
 - EGG ROLL
 - Egg  Roll
 - FISH ROLL
 - FRESH YOGHURT BOTTLE
 - Fish  Roll
 - Fresh  Yoghurt  Bottle
 - MEAT PIE
 - Meat  Pie
 - PUFF PUFF PACK
 - Puff  Puff  Pack
 - RED VELVET SLICE
 - Red  Velvet  Slice
 - SAUSAGE ROLL
 - SLICED SANDWICH BREAD
 - SOFT DRINK
 - Sandwich Bread
 - Sausage  Roll
 - Sausage Rol
 - Sliced  Sandwich  Bread
 - Sof

In [4]:
# ============================================================
# STAGE 2 — STEP 3
# PRODUCT NAME STANDARDIZATION
# ============================================================

print("=" * 70)
print("STAGE 2 — PRODUCT NAME STANDARDIZATION")
print("=" * 70)


# ============================================================
# 1. DISPLAY MASTER PRODUCTS
# ============================================================

print("\nAUTHORITATIVE PRODUCT MASTER")

master_products = (
    products_clean["product_name"]
    .dropna()
    .unique()
    .tolist()
)

for product in master_products:
    print(" -", product)


# ============================================================
# 2. CREATE STANDARDIZATION FUNCTION
# ============================================================

def normalize_product_name(value):

    if pd.isna(value):
        return value

    value = str(value)

    # Remove leading/trailing spaces
    value = value.strip()

    # Collapse multiple spaces
    value = " ".join(value.split())

    # Case-insensitive matching
    value_lower = value.lower()

    # Explicit corrections
    corrections = {

        "agege bread":
            "Agege Bread",

        "banana cake loaf":
            "Banana Cake Loaf",

        "birthday cake 8in":
            "Birthday Cake 8in",

        "bottled water":
            "Bottled Water",

        "chicken pie":
            "Chicken Pie",

        "chiken pie":
            "Chicken Pie",

        "chin chin pack":
            "Chin Chin Pack",

        "coconut bread":
            "Coconut Bread",

        "cream cake slice":
            "Cream Cake Slice",

        "cupcake":
            "Cupcake",

        "doughnut":
            "Doughnut",

        "donut":
            "Doughnut",

        "egg roll":
            "Egg Roll",

        "fish roll":
            "Fish Roll",

        "fresh yoghurt bottle":
            "Fresh Yoghurt Bottle",

        "yogurt bottle":
            "Fresh Yoghurt Bottle",

        "meat pie":
            "Meat Pie",

        "puff puff pack":
            "Puff Puff Pack",

        "red velvet slice":
            "Red Velvet Slice",

        "sausage roll":
            "Sausage Roll",

        "sausage rol":
            "Sausage Roll",

        "sliced sandwich bread":
            "Sliced Sandwich Bread",

        "sandwich bread":
            "Sliced Sandwich Bread",

        "soft drink":
            "Soft Drink",

        "zobo bottle":
            "Zobo Bottle"
    }

    return corrections.get(
        value_lower,
        value
    )


# ============================================================
# 3. KEEP ORIGINAL PRODUCT NAME
# ============================================================

transactions_clean[
    "product_name_original"
] = transactions_clean[
    "product_name"
]


# ============================================================
# 4. STANDARDIZE TRANSACTION PRODUCTS
# ============================================================

transactions_clean[
    "product_name"
] = transactions_clean[
    "product_name"
].apply(normalize_product_name)


# ============================================================
# 5. STANDARDIZE PRODUCTION PRODUCTS
# ============================================================

production_clean[
    "product_name_original"
] = production_clean[
    "product_name"
]

production_clean[
    "product_name"
] = production_clean[
    "product_name"
].apply(normalize_product_name)


# ============================================================
# 6. CHECK AGAINST MASTER PRODUCT LIST
# ============================================================

master_products = set(
    products_clean["product_name"]
    .dropna()
)

transaction_products = set(
    transactions_clean["product_name"]
    .dropna()
)

unknown_products = (
    transaction_products -
    master_products
)


print("\n" + "=" * 70)
print("POST-STANDARDIZATION VALIDATION")
print("=" * 70)

print(
    "\nUnique transaction products:",
    len(transaction_products)
)

print(
    "Master products:",
    len(master_products)
)

print(
    "Unknown products:",
    len(unknown_products)
)


# ============================================================
# 7. DISPLAY UNKNOWN PRODUCTS
# ============================================================

if unknown_products:

    print("\nProducts still not in master table:")

    for product in sorted(unknown_products):

        print(" -", product)

else:

    print(
        "\n✓ Every transaction product "
        "matches the product master."
    )


# ============================================================
# 8. PRODUCT NAME TRANSFORMATION SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PRODUCT NAME TRANSFORMATION")
print("=" * 70)

comparison = (
    transactions_clean[
        [
            "product_name_original",
            "product_name"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "product_name",
            "product_name_original"
        ]
    )
)

display(comparison)


# ============================================================
# 9. TRANSACTION COUNTS BY STANDARD PRODUCT
# ============================================================

print("\n" + "=" * 70)
print("TRANSACTIONS BY STANDARDIZED PRODUCT")
print("=" * 70)

product_counts = (
    transactions_clean[
        "product_name"
    ]
    .value_counts()
    .rename_axis("product_name")
    .reset_index(name="transaction_count")
)

display(product_counts)


# ============================================================
# 10. CHECK TEST ITEM
# ============================================================

test_item_count = (
    transactions_clean[
        transactions_clean["product_name"]
        .str.lower()
        .eq("test item")
    ]
    .shape[0]
)

print(
    f"\nTEST ITEM records: {test_item_count:,}"
)

if test_item_count > 0:

    print(
        "\nWARNING: TEST ITEM does not exist "
        "in the product master."
    )

    display(
        transactions_clean[
            transactions_clean["product_name"]
            .str.lower()
            .eq("test item")
        ].head(20)
    )


# ============================================================
# 11. SAVE INTERMEDIATE CLEAN DATA
# ============================================================

transactions_clean.to_csv(
    PROCESSED_DIR / "transactions_clean.csv",
    index=False
)

production_clean.to_csv(
    PROCESSED_DIR / "production_log_clean.csv",
    index=False
)

print("\n" + "=" * 70)
print("PRODUCT STANDARDIZATION COMPLETE")
print("=" * 70)

print(
    "\nUpdated files saved to:",
    PROCESSED_DIR
)

STAGE 2 — PRODUCT NAME STANDARDIZATION

AUTHORITATIVE PRODUCT MASTER
 - Agege Bread
 - Sliced Sandwich Bread
 - Coconut Bread
 - Meat Pie
 - Chicken Pie
 - Sausage Roll
 - Doughnut
 - Egg Roll
 - Fish Roll
 - Puff Puff Pack
 - Chin Chin Pack
 - Birthday Cake 8in
 - Cream Cake Slice
 - Cupcake
 - Banana Cake Loaf
 - Red Velvet Slice
 - Zobo Bottle
 - Fresh Yoghurt Bottle
 - Bottled Water
 - Soft Drink

POST-STANDARDIZATION VALIDATION

Unique transaction products: 21
Master products: 20
Unknown products: 1

Products still not in master table:
 - TEST ITEM

PRODUCT NAME TRANSFORMATION


,product_name_original,product_name
784,AGEGE BREAD,Agege Bread
121,Agege Bread,Agege Bread
3,Agege Bread,Agege Bread
469,agege bread,Agege Bread
3223,BANANA CAKE LOAF,Banana Cake Loaf
...,...,...
1668,TEST ITEM,TEST ITEM
1255,ZOBO BOTTLE,Zobo Bottle
564,Zobo Bottle,Zobo Bottle
73,Zobo Bottle,Zobo Bottle



TRANSACTIONS BY STANDARDIZED PRODUCT


,product_name,transaction_count
0,Meat Pie,7281
1,Agege Bread,6351
2,Sausage Roll,5471
3,Bottled Water,5271
4,Doughnut,4888
5,Soft Drink,4650
6,Sliced Sandwich Bread,4642
7,Chicken Pie,4466
8,Fish Roll,3586
9,Puff Puff Pack,3549



TEST ITEM records: 12



,transaction_id,timestamp,product_name,quantity,unit_price,discount_pct,payment_method,cashier,is_negative_quantity,product_name_original
1668,TEST007,2026-03-05 17:46:43,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
6407,TEST003,2026-03-18 14:55:41,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
14151,TEST005,2026-04-07 15:41:13,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
15641,TEST011,2026-04-11 14:48:39,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
29889,TEST010,2026-05-19 18:19:30,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
33763,TEST006,2026-05-29 19:16:06,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
34839,TEST000,2026-06-01 11:45:16,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
40821,TEST009,2026-06-17 11:05:27,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
47681,TEST004,2026-07-04 15:50:39,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM
54075,TEST008,2026-07-22 07:04:40,TEST ITEM,1,1.0,0.0,Cash,Ada,False,TEST ITEM



PRODUCT STANDARDIZATION COMPLETE

Updated files saved to: c:\Users\user\Desktop\Bakery dashboard\data\processed
